## SECTION 2.1 - SMOLAGENTS FRAMEWORK PRACTICE

#### Table of Content:
- Question 2 (Q2)
- Question 3 (Q3)
- Question 4 (Q4)

### Q2

* **Challenge:**
Set Up a Multi-Agent System with Manager and Web Search Agents

* **Assessment Criteria:**
    * Web agent has correct tools configured
    * Manager agent properly references web agent
    * Appropriate max_steps value is set
    * Required imports are authorized


        ```
        # Create web agent and manager agent structure
        web_agent = ToolCallingAgent(
        tools=[],           # Add required tools
        model=None,         # Add model
        max_steps=5,        # Adjust steps
        name="",           # Add name
        description=""      # Add description
        )

        manager_agent = CodeAgent()
        ```

* **Solution:**
```
web_agent = ToolCallingAgent(
    tools=[DuckDuckGoSearchTool(), visit_webpage],
    model=model,
    max_steps=10,
    name="search",
    description="Runs web searches for you."
)

manager_agent = CodeAgent(
    tools=[],
    model=model,
    managed_agents=[web_agent],
    additional_authorized_imports=["time", "numpy", "pandas"]
)
```

* **Q2 Bonus:**

In [ ]:
from smolagents.agent import ToolCallingAgent, CodeAgent
from smol_dev import prompt_ui
from smol_dev.prompts import plan_task
from typing import List
from smolagents.tools import WebSearchTool  # Correct import

# Create web search tool
web_search_tool = WebSearchTool()

# Create web agent with the web search tool
web_agent = ToolCallingAgent(
    tools=[web_search_tool],
    model="gpt-4-1106-preview",  # Or another suitable model
    max_steps=8,  # Adjusted max_steps to be potentially higher, but reasonable.  5 might be too low.
    name="WebSearcher",
    description="An agent capable of searching the web using a search engine to find information."
)

# Create manager agent, referencing and delegating to the web agent
class ManagerAgent(CodeAgent):  # Inherit from CodeAgent, it gives a solid starting point.
    def __init__(self, web_agent: ToolCallingAgent):
        super().__init__()
        self.web_agent = web_agent # Store the web agent for later use
        self.prompt_ui = prompt_ui
        self.plan_task = plan_task

    def process_response(self, prompt: str, previous_messages=None) -> str:

        """Override the standard CodeAgent process_response to include delegation.
           The method handles:
           1. Initial Task Planning: creating or getting the inital plan/goal.
           2. Deciding to Delegate, calling websearch, or directly answering.
           3. Tracking the state of previous_messages to guide long conversations,
           4. A hardcoded limit on iteration/recursion depth in a single run of the agent.
        Args:
            prompt (str): The incoming user prompt.
            previous_messages: Prior messages in the conversation for context continuity.

        Returns:
            str: The agent's response, either from delegation or its direct output.
        """
        if previous_messages is None:
            # if no messages have been sent, assume base message.
            previous_messages = []

        if not previous_messages: # If it's the first message
            # Plan the task
            plan = self.prompt_ui(prompt=self.plan_task, messages=previous_messages)
            previous_messages.append({"role": "assistant", "content": plan})
            previous_messages.append({"role": "user", "content": prompt})

        # Decide whether to delegate or answer directly
        delegation_check_prompt = f"""
        Based on the user query and current conversation, should you delegate to the web search agent? 
        Consider these questions: 
        1.Is this a straightforward code request, like generating simple functions or classes that don't need external data?  -> NO
        2. Does the user explicitly ask for online resources, documentation, or external real-time information? ->YEs
        3. If this were a multi-step task involving coding *AND* finding info, is it already been handled by web resarch? -> NO

        
        Current Conversation: {previous_messages}
        
        Answer with YES or NO.
        """

        should_delegate = self.prompt_ui(prompt=delegation_check_prompt, messages=previous_messages)
        should_delegate = should_delegate.strip().upper()  # Remove leading/trailing spaces and force uppercase

        if "YES" in should_delegate:
             # Delegate to the web agent
            web_search_response = self.web_agent.process_response(prompt, previous_messages)
            previous_messages.append({"role": "web_agent", "content": web_search_response})  # Keep for history.
            # Combine and add to the converstaion for context in the next loop
            combined_response = f"The web search returned:\n{web_search_response}\n\nHow should I use this info to update or generate code?"
            return combined_response

        elif "NO" in should_delegate:
            code_generation_prompt = f"{previous_messages}\nBased on the conversation so far, generate the code"
            code_response = self.prompt_ui(prompt=code_generation_prompt, messages=previous_messages)  # Or a different prompt.
            previous_messages.append({"role": "assistant", "content": code_response})
            return code_response

        else:
            # Handle cases where the YES/NO response is unclear.
            return "I'm unable to determine the best course of action. Please rephrase your request."

# Create Manager Agent and pass Web Agent to it
manager_agent = ManagerAgent(web_agent=web_agent)

# Example usage:
user_prompt = "Write a Python function to calculate the factorial of a number. Can you look up recent discussions about optimizations for factorial functions for large numbers?"
response = manager_agent.process_response(user_prompt)
print(f"Manager Agent Response: {response}")

user_prompt2 = "Use this information to create a more efficient version"
response2 = manager_agent.process_response(user_prompt2, previous_messages=[{"role": "user", "content": user_prompt}, {"role": "assistant", "content": response}]) # VERY important to add correct message history
print(f"Manager Agent Response (2): {response2}")


user_prompt3 = "write a function that takes two lists and returns a new list containing elements that are commonly present in both input lists"
response3 = manager_agent.process_response(user_prompt3, previous_messages=[
    {"role": "user", "content": user_prompt},
    {"role": "assistant", "content": response},
    {"role": "user", "content": user_prompt2},
    {"role": "assistant", "content": response2}
])  # VERY important to add correct message history
print(f"Manager Agent Response (3): {response3}")

### Q3

* **Challenge:**
Configure Agent Security Settings

* **Assessment Criteria:**
    * E2B sandbox is properly configured
    * Authorized imports are appropriately limited
    * Security settings are correctly implemented
    * Basic agent configuration is maintained


        ```
        # Set up secure code execution environment
        from smolagents import CodeAgent

        agent = CodeAgent(
        tools=[],
        model=model
        # Add security configuration
        )
        ```

* **Solution:**
```
from smolagents import CodeAgent, E2BSandbox

agent = CodeAgent(
    tools=[],
    model=model,
    sandbox=E2BSandbox(),
    additional_authorized_imports=["numpy"]
)
```

* **Q3 Bonus:**

In [ ]:
from smolagents.agent import CodeAgent
from smol_dev import prompt_ui
from e2b import Sandbox
from smolagents.llms import HfApiModel  


# Define authorized imports (Important for security)
AUTHORIZED_IMPORTS = {
    "allowed_packages": [
        "math",
        "datetime",
        "json",
        "requests",
    ],
    "allowed_modules": [],
}

# Define security settings
SECURITY_SETTINGS = {
    "timeout": 60,
    "ram_mb": 512,
    "cpu_count": 1,
    "disk_mb": 100,
    "on_tool_error": 'exit',
    "available_process_ports": [80, 443, 8080]
}


class SecureCodeAgent(CodeAgent):
    def __init__(self, model, **kwargs):  # Remove default model value
        super().__init__(model=model, **kwargs)
        self.prompt_ui = prompt_ui

    def _run_code_in_sandbox(self, code: str) -> str:
        """Runs Python code within a secure E2B sandbox with specified limits.
        """

        try:
            with Sandbox(
                template="base-python-template",
                timeout=SECURITY_SETTINGS["timeout"],
                memory_mb=SECURITY_SETTINGS["ram_mb"],
                cpu_count=SECURITY_SETTINGS["cpu_count"],
                extra_specs = {
                    "disk_size": SECURITY_SETTINGS["disk_mb"]
                    }
            ) as sandbox:
                # Set up authorized imports within the sandbox
                sandbox.filesystem.write("/smol_autorized_modules.json", str(AUTHORIZED_IMPORTS))

                # Execute the code in the sandbox
                process = sandbox.process.start_and_wait(
                    f'python -c "{code}"',
                    on_stdout=print,
                    on_stderr=print,
                    timeout=SECURITY_SETTINGS["timeout"],
                )


                if process.exit_code != 0:
                     return f"Error: Code execution failed. Exit code: {process.exit_code}\n stderr {process.stderr}"

                output = process.stdout
                if output and isinstance(output, list):
                    output = "\n".join(output)

                return output or ""

        except Exception as e:
             return  f"Sandbox Error: {e}"

# Instantiate the secure agent with HfApiModel
model = HfApiModel(model_id="Qwen/Qwen2.5-Coder-32B-Instruct")  # Use HfApiModel
agent = SecureCodeAgent(model=model)  # Pass the model instance


# Example usage
code_to_execute = """
import math
print(math.factorial(5))
"""
result = agent._run_code_in_sandbox(code_to_execute)
print(f"Result: {result}")


### Q4

* **Challenge:**
Implement a Tool-Calling Agent

* **Assessment Criteria:**
    * Tools are properly configured
    * Step limit is set appropriately
    * Agent name and description are provided
    * Basic configuration is complete


        ```
        # Create a tool-calling agent
        from smolagents import ToolCallingAgent

        agent = ToolCallingAgent(
        # Add configuration here
        )
        ```

* **Solution:**
```
from smolagents import ToolCallingAgent

agent = ToolCallingAgent(
    tools=[custom_tool],
    model=model,
    max_steps=5,
    name="tool_agent",
    description="Executes specific tools based on input"
)
```

* **Q4 Bonus:**

In [ ]:
from smolagents.agent import ToolCallingAgent
from smolagents.tools import WebSearchTool, PythonREPLTool  # Import useful tools

# Create instances of the tools
web_search_tool = WebSearchTool()
python_repl_tool = PythonREPLTool()

# Create the ToolCallingAgent
agent = ToolCallingAgent(
    tools=[web_search_tool, python_repl_tool],  # Include the tools
    model="gpt-4-1106-preview",  # Specify a capable model (must support function calling)
    max_steps=10,            # Set a reasonable step limit
    name="MultiToolAgent",       # Give the agent a name
    description="An agent that can use both web search and a Python REPL.",  # Add a description
)

# Example usage: Interacting with the agent
user_prompt = "What is the square root of 144?  And also, what is the current capital of France?"

# Process the user's prompt
response = agent.process_response(user_prompt)  # Call the process_response function
print(response)

user_prompt_2 = "Can you find me information on the latest advancements in AI?"
response_2 = agent.process_response(user_prompt_2)
print(response_2)

user_prompt_3 = "What is the result of 234234 / 23 ?"
response_3 = agent.process_response(user_prompt_3)
print(response_3)

* ***Q4 Bonus - 1 (if HfApiModel is preferred):***

In [ ]:
from smolagents.agent import ToolCallingAgent
from smolagents.tools import WebSearchTool, PythonREPLTool  # Import tools
from smolagents.llms import HfApiModel  

# Create instances of the tools
web_search_tool = WebSearchTool()
python_repl_tool = PythonREPLTool()

# Create the HfApiModel instance
model = HfApiModel(model_id="Qwen/Qwen2.5-Coder-32B-Instruct")

# Create the ToolCallingAgent
agent = ToolCallingAgent(
    tools=[web_search_tool, python_repl_tool],  # Tool instances
    model=model,  # Use the HfApiModel instance
    max_steps=10,
    name="MultiToolAgent",
    description="An agent that can use both web search and a Python REPL.",
)

# Example usage (minimal, just to show it's set up)
user_prompt = "What is the capital of Australia?"
response = agent.process_response(user_prompt)
print(response)